# Practice Lab: ViT & CLIP (Lesson 6.4)

Module 6 · 8 exercises
**Needs:** torch, transformers, timm, faiss-cpu

Exercises 1 (patches) and 3 (contrastive loss) run locally.


# Lesson 6.4: Vision Transformers (ViT) & CLIP

8 exercises: 2E + 3M + 3C
**Needs:** `pip install torch torchvision transformers timm faiss-cpu`

Exercises 1 (patch math) and 3 (contrastive loss) run **locally** with PyTorch.


In [ ]:
!pip install torch torchvision transformers timm faiss-cpu matplotlib -q


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math


---
## Exercise 1 (Easy): ViT Patch Visualization

Split 224x224 image into 16x16 patches. Verify dimensions.


In [ ]:
# YOUR CODE
# TODO: create image, unfold into patches
# TODO: print patch count, dimensions, [CLS] token count


<details><summary>Solution</summary>


In [ ]:
import torch

# Synthetic image
img = torch.randn(3, 224, 224)
patch_size = 16

# Method 1: unfold
patches = img.unfold(1, patch_size, patch_size
                    ).unfold(2, patch_size, patch_size)
n_h, n_w = patches.shape[1], patches.shape[2]
n_patches = n_h * n_w

print('ViT Patch Calculation:')
print(f'  Image size: {img.shape[1]}x{img.shape[2]}')
print(f'  Patch size: {patch_size}x{patch_size}')
print(f'  Grid: {n_h}x{n_w} = {n_patches} patches')
print(f'  Each patch pixels: {patch_size}x{patch_size}'
      f' = {patch_size**2}')
print(f'  Flattened (with channels): '
      f'{3}x{patch_size**2} = {3*patch_size**2} dims')
print(f'  With [CLS] token: {n_patches} + 1 = '
      f'{n_patches + 1} tokens')
print()

# Method 2: Conv2d projection (how ViT actually does it)
proj = nn.Conv2d(3, 768, kernel_size=patch_size,
                 stride=patch_size)
projected = proj(img.unsqueeze(0))
# Shape: (1, 768, 14, 14) -> flatten -> (1, 196, 768)
tokens = projected.flatten(2).transpose(1, 2)
print(f'Conv2d projection:')
print(f'  Input: {img.unsqueeze(0).shape}')
print(f'  After conv: {projected.shape}')
print(f'  Tokens: {tokens.shape}')
print(f'  Each token: {tokens.shape[2]}-dim vector')

# Verify different image sizes
for size in [224, 384, 512]:
    n = (size // patch_size) ** 2
    print(f'  {size}x{size} -> {n} patches '
          f'({size//patch_size}x{size//patch_size})')

# Verify: patch_size 14 vs 16
for ps in [14, 16, 32]:
    n = (224 // ps) ** 2
    print(f'  patch_size={ps}: {n} patches '
          f'({224//ps}x{224//ps})')


</details>


---
## Exercise 3 (Medium): Contrastive Loss Implementation

InfoNCE loss from scratch. Verify with perfect vs random alignment.


In [ ]:
# YOUR CODE
# TODO: implement clip_loss
# TODO: test perfect, random, temperature sweep


<details><summary>Solution</summary>


In [ ]:
import torch
import torch.nn.functional as F

def clip_loss(img_emb, txt_emb, temperature=0.07):
    img = F.normalize(img_emb, dim=-1)
    txt = F.normalize(txt_emb, dim=-1)
    logits = (img @ txt.T) / temperature
    labels = torch.arange(len(logits))
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2

N, D = 8, 128
print('Contrastive Loss (InfoNCE):')
print(f'  Batch size: {N}, Embedding dim: {D}')
print()

# Test 1: Perfect alignment
shared = F.normalize(torch.randn(N, D), dim=-1)
loss_perfect = clip_loss(shared, shared)
print(f'  Perfect alignment: {loss_perfect:.6f}')
print(f'    (should be ~0.0)')

# Test 2: Random (no alignment)
img_rand = torch.randn(N, D)
txt_rand = torch.randn(N, D)
loss_random = clip_loss(img_rand, txt_rand)
expected_random = math.log(N)
print(f'  Random alignment:  {loss_random:.4f}')
print(f'    (expected ~ln({N}) = {expected_random:.4f})')

# Test 3: Partial alignment (50% correct)
img_partial = torch.randn(N, D)
txt_partial = torch.randn(N, D)
txt_partial[:N//2] = img_partial[:N//2] + 0.1 * torch.randn(N//2, D)
loss_partial = clip_loss(img_partial, txt_partial)
print(f'  Partial alignment: {loss_partial:.4f}')
print(f'    (between perfect and random)')

# Verify ordering
assert loss_perfect < loss_partial < loss_random, 'Order wrong!'
print(f'\n  Ordering verified: perfect < partial < random')

# Temperature sweep
print(f'\n  Temperature sweep (random embeddings):')
for temp in [0.01, 0.07, 0.1, 0.5, 1.0]:
    loss = clip_loss(img_rand, txt_rand, temp)
    print(f'    tau={temp:.2f}: loss={loss:.4f}')

# Batch size effect on difficulty
print(f'\n  Batch size effect (random):')
for n in [4, 8, 16, 32, 64]:
    ie = torch.randn(n, D)
    te = torch.randn(n, D)
    loss = clip_loss(ie, te)
    print(f'    N={n:3d}: loss={loss:.4f} '
          f'(ln({n})={math.log(n):.4f})')
print(f'\n  Key insight: random loss ~ ln(N)')
print(f'  Larger batches = harder task = better learning')


</details>


---
## Exercise 2 (Easy): CLIP Zero-Shot Classifier

Classify Indian categories with CLIP text prompts, no training. See practice-lab-6_4.html.

*Needs `transformers` + CLIP model download (runs on Colab CPU).*

In [ ]:
# YOUR CODE
# TODO: load CLIP ViT-B/32 (transformers)
# TODO: define 5 Indian categories (food, temple, traffic, nature, festival)
# TODO: classify 10 images, print prediction + confidence
# TODO: compute overall accuracy

<details><summary>Solution</summary>

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch, requests

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32")

categories = ["Indian food", "Hindu temple",
              "Indian traffic", "nature landscape",
              "Indian festival celebration"]

def classify(image, categories):
    prompts = [f"a photo of {c}" for c in categories]
    inputs = processor(text=prompts, images=image,
                       return_tensors="pt", padding=True)
    with torch.no_grad():
        probs = model(**inputs).logits_per_image[0
                     ].softmax(dim=-1)
    results = {c: p.item()
               for c, p in zip(categories, probs)}
    best = max(results, key=results.get)
    return best, results[best], results

# Test with sample images
# TODO: load 10 images (URLs or local)
# TODO: classify each, print prediction + confidence
# TODO: calculate accuracy

</details>

---
## Exercise 4 (Medium): CLIP Limitation Test

Probe CLIP's 5 known failure modes. See practice-lab-6_4.html.

*Needs `transformers` + CLIP model download.*

In [ ]:
# YOUR CODE
# TODO: build test cases for CLIP's 5 failure modes
#   (counting, spatial, compositionality, typographic, bias)
# TODO: run CLIP similarity on each, record which reproduce
# TODO: explain how each failure affects Stable Diffusion

<details><summary>Solution</summary>

In [ ]:
# Test each failure mode
# 1. Counting: "3 dogs" vs "5 dogs" on same image
# 2. Spatial: "cat on table" vs "cat under table"
# 3. Compositionality: "red cube blue sphere"
#    vs "blue cube red sphere"
# 4. Typographic: write "cat" text on a dog photo
# 5. Bias: test with cultural stereotypes

# For each: compute CLIP similarities and
# check if CLIP can distinguish the cases

# TODO: implement each test
# TODO: document failure severity
# TODO: explain SD implications

</details>

---
## Exercise 5 (Medium): Image Search with FAISS

CLIP + FAISS image search over a synthetic gallery. See practice-lab-6_4.html.

*Needs `transformers`, `faiss-cpu` + CLIP model download.*

In [ ]:
# YOUR CODE
# TODO: create 20+ synthetic images (solid colors / gradients)
# TODO: encode with CLIP, L2-normalize, build a FAISS IndexFlatIP
# TODO: search with 10 text queries, inspect top-K results

<details><summary>Solution</summary>

In [ ]:
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch, numpy as np
import faiss

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32")

# Create color gallery
colors = {
    "red": (255,0,0), "blue": (0,100,255),
    "green": (0,180,0), "yellow": (255,230,0),
    "orange": (255,140,0), "purple": (128,0,255),
    "white": (255,255,255), "black": (10,10,10),
    "cyan": (0,200,200), "pink": (255,150,180),
}

# Encode images
embeddings = []
names = list(colors.keys())
for name, rgb in colors.items():
    img = Image.new("RGB", (224, 224), rgb)
    inputs = processor(images=img,
                       return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    embeddings.append(emb[0].numpy())

# Build FAISS index
emb_array = np.stack(embeddings).astype("float32")
index = faiss.IndexFlatIP(emb_array.shape[1])
index.add(emb_array)

# Search with text
queries = ["ocean waves", "fire flames",
           "forest trees", "sunshine bright",
           "night sky"]

for q in queries:
    inputs = processor(text=[q], return_tensors="pt")
    with torch.no_grad():
        qe = model.get_text_features(**inputs)
    qe = qe / qe.norm(dim=-1, keepdim=True)
    D, I = index.search(qe.numpy().astype("float32"), 3)
    top3 = [f"{names[i]}({D[0][j]:.2f})"
            for j, i in enumerate(I[0])]
    print(f"  '{q}' -> {', '.join(top3)}")

</details>

---
## Exercise 6 (Challenge): CLIP Score Evaluator

Score generated images against prompts with CLIP Score. See practice-lab-6_4.html.

*Needs `torchmetrics` + Stable Diffusion outputs.*

In [ ]:
# YOUR CODE
# TODO: generate 20 images from 5 prompts with Stable Diffusion
# TODO: compute CLIP Score for each image-prompt pair
# TODO: rank prompts by average CLIP Score
# TODO: compare with your own quality judgment

<details><summary>Solution</summary>

In [ ]:
# pip install torchmetrics
from torchmetrics.multimodal import CLIPScore
import torch

metric = CLIPScore(model_name_or_path=
    "openai/clip-vit-base-patch32")

# TODO: generate 20 images with SD
# TODO: compute CLIP Score for each
# TODO: rank prompts by average score
# TODO: compare with human judgment

</details>

---
## Exercise 7 (Challenge): Fine-tune ViT with timm

Linear probe vs full fine-tune on CIFAR-100. See practice-lab-6_4.html.

*Needs `timm` + GPU recommended.*

In [ ]:
# YOUR CODE
# TODO: load vit_base_patch16_224 from timm (num_classes=10)
# TODO: pick 10 CIFAR-100 classes
# TODO: train a linear probe (frozen backbone), record accuracy
# TODO: full fine-tune, record accuracy, compare

<details><summary>Solution</summary>

In [ ]:
import timm
import torch, torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Load pre-trained ViT
model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True, num_classes=10)

# Strategy 1: freeze backbone (linear probe)
for name, param in model.named_parameters():
    if 'head' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters()
                if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Linear probe: {trainable:,} / {total:,} "
      f"trainable ({trainable/total*100:.2f}%)")

# TODO: load CIFAR-100, select 10 classes
# TODO: train linear probe, record accuracy
# TODO: unfreeze all, full fine-tune
# TODO: compare accuracies

</details>

---
## Exercise 8 (Challenge): Indian E-Commerce Visual Search

CLIP + FAISS product search over an Indian catalog. See practice-lab-6_4.html.

*Needs `transformers`, `faiss-cpu` + CLIP model download.*

In [ ]:
# YOUR CODE
# TODO: build a catalog of 20+ Indian product descriptions
# TODO: encode with CLIP text encoder, L2-normalize, FAISS index
# TODO: search 'red Banarasi saree', 'brass temple bell', etc.
# TODO: compare Hindi vs English queries

<details><summary>Solution</summary>

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch, numpy as np, faiss

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32")

products = [
    "Red Banarasi silk saree with gold zari border",
    "Blue Kanjeevaram silk saree with temple motif",
    "White cotton kurta with chikankari embroidery",
    "Gold jhumka earrings with pearl drops",
    "Brass Ganesh idol for home decor",
    "Kolhapuri leather chappal, brown",
    "Madhubani painting of Radha Krishna",
    "Steel masala dabba with 7 containers",
    "Handloom Pochampally ikat dupatta",
    "Terracotta Diwali diya set of 12",
    # TODO: add 10 more products
]

# Encode products
embs = []
for desc in products:
    inputs = processor(text=[desc],
                       return_tensors="pt")
    with torch.no_grad():
        e = model.get_text_features(**inputs)
    e = e / e.norm(dim=-1, keepdim=True)
    embs.append(e[0].numpy())

# Build FAISS index
arr = np.stack(embs).astype("float32")
index = faiss.IndexFlatIP(arr.shape[1])
index.add(arr)

# Search
queries = [
    "red silk saree for wedding",
    "brass temple bell",
    "leather sandals",
    "traditional painting",
    "kitchen spice container",
]

print("Product Search Results:")
for q in queries:
    inp = processor(text=[q], return_tensors="pt")
    with torch.no_grad():
        qe = model.get_text_features(**inp)
    qe = qe / qe.norm(dim=-1, keepdim=True)
    D, I = index.search(
        qe.numpy().astype("float32"), 3)
    print(f"\n  Query: '{q}'")
    for j, i in enumerate(I[0]):
        print(f"    {j+1}. [{D[0][j]:.3f}] "
              f"{products[i][:50]}...")

</details>